# Set up environment

⚠️ Must restart session after installing HMMER package in order for ANARCI to install and run correctly.

⏳ 5 minutes

In [1]:
# @title Mount Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# @title Install Packages

!pip install 'transformers<5.0.0' # antiBERTy embeddings require compatible transformers version
!pip install antiberty #required for antiBERTy embeedings
!pip install tensorflow #required for neural networks
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install tensorflow
!pip install matplotlib
!pip install scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 MB 7.3 MB/s eta 0:00:00


In [3]:
# @title Load Libraries

# ===== Data tools =====
import pandas as pd
import numpy as np
import csv
import os
import gc
import joblib

# ===== Visualization =====
import matplotlib.pyplot as plt
import seaborn as sns

# ===== Deep learning tools =====
import torch
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# ===== Embeddings tools =====
from antiberty import AntiBERTyRunner

# ===== Preprocessing & Evaluation =====
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

# ===== Models =====
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

# AntiBERTy embedddings

Goal: embeddings of aligned sequences

⏳ 7 minutes

In [4]:
# @title Create a working directory on Drive
output_dir = "/content/drive/MyDrive/antiberty_antibody_results"
!mkdir -p {output_dir}
print(f"Results will be saved to: {output_dir}")

Results will be saved to: /content/drive/MyDrive/antiberty_antibody_results


In [5]:
# @title AntiBERTy embeddings for VH sequences

# ===== Parameters =====
csv_file = "/content/drive/MyDrive/poster_antibodies/merged_anarci_vh.csv"
prefix = "vh_antiberty"
length = 127
processor = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_dir = "/content/drive/MyDrive/antiberty_antibody_results"

# Load CSV
df = pd.read_csv(csv_file)

# Padding function
def pad_sequence(seq, template_len):
    return seq.ljust(template_len, "-")[:template_len]

df["aligned_imgt_padded"] = df["aligned_imgt"].apply(
    lambda x: pad_sequence(x.strip('"').strip(), length)
)

# ===== AntiBERTy Embeddings =====
try:
    from antiberty import AntiBERTyRunner
except ImportError:
    !pip install antiberty -q
    from antiberty import AntiBERTyRunner

antiberty = AntiBERTyRunner()
emb_size = 512  # AntiBERTy fixed embedding dimension

# Embedding function (corrected)
def embed_sequence(seq):
    mask = torch.tensor([aa != "-" for aa in seq], device=processor)
    seq_no_gaps = "".join([aa for aa in seq if aa != "-"])

    if len(seq_no_gaps) == 0:
        per_residue = torch.zeros((length, emb_size))
        per_sequence = torch.zeros((emb_size,))
        return per_residue, per_sequence

    with torch.no_grad():
        emb_tensor = antiberty.embed([seq_no_gaps])[0]          # (L+2, 512)
        residue_emb = emb_tensor[1:-1, :].to(processor)         # (L, 512)

    per_residue = torch.zeros((length, emb_size), device=processor)
    per_residue[mask] = residue_emb
    per_sequence = per_residue[mask].mean(0)

    del emb_tensor, residue_emb
    gc.collect()
    return per_residue.cpu(), per_sequence.cpu()


# Run embedding
N = len(df)
X_residue = np.zeros((N, length, emb_size), dtype=np.float32)
X_sequence = np.zeros((N, emb_size), dtype=np.float32)
antibody_ids = []

for i, (ab_id, seq) in enumerate(zip(df["antibody_id"], df["aligned_imgt_padded"])):
    print(f"Embedding {i+1}/{N}: {ab_id}")
    per_res, per_seq = embed_sequence(seq)
    X_residue[i] = per_res.numpy()
    X_sequence[i] = per_seq.numpy()
    antibody_ids.append(ab_id)

# Save outputs
os.makedirs(output_dir, exist_ok=True)
np.save(os.path.join(output_dir, f"{prefix}_per_residue.npy"), X_residue)
np.save(os.path.join(output_dir, f"{prefix}_per_sequence.npy"), X_sequence)
np.save(os.path.join(output_dir, f"{prefix}_antibody_ids.npy"), np.array(antibody_ids))
np.save(os.path.join(output_dir, f"{prefix}_imgt_positions.npy"), np.arange(1, length + 1))

print("Embeddings for VH domain saved successfully")

Embedding 1/208: GDPa1-001
Embedding 2/208: GDPa1-002
Embedding 3/208: GDPa1-003
Embedding 4/208: GDPa1-004
Embedding 5/208: GDPa1-005
Embedding 6/208: GDPa1-006
Embedding 7/208: GDPa1-007
Embedding 8/208: GDPa1-008
Embedding 9/208: GDPa1-010
Embedding 10/208: GDPa1-011
Embedding 11/208: GDPa1-012
Embedding 12/208: GDPa1-014
Embedding 13/208: GDPa1-015
Embedding 14/208: GDPa1-016
Embedding 15/208: GDPa1-017
Embedding 16/208: GDPa1-018
Embedding 17/208: GDPa1-019
Embedding 18/208: GDPa1-020
Embedding 19/208: GDPa1-021
Embedding 20/208: GDPa1-022
Embedding 21/208: GDPa1-023
Embedding 22/208: GDPa1-024
Embedding 23/208: GDPa1-027
Embedding 24/208: GDPa1-028
Embedding 25/208: GDPa1-029
Embedding 26/208: GDPa1-030
Embedding 27/208: GDPa1-031
Embedding 28/208: GDPa1-032
Embedding 29/208: GDPa1-033
Embedding 30/208: GDPa1-034
Embedding 31/208: GDPa1-035
Embedding 32/208: GDPa1-036
Embedding 33/208: GDPa1-037
Embedding 34/208: GDPa1-038
Embedding 35/208: GDPa1-040
Embedding 36/208: GDPa1-042
E

In [6]:
# @title AntiBERTy embeddings for VL sequences

# ===== Parameters =====
csv_file = "/content/drive/MyDrive/poster_antibodies/merged_anarci_vl.csv"
prefix = "vl_antiberty"
length = 127
processor = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_dir = "/content/drive/MyDrive/antiberty_antibody_results"

# Load CSV
df = pd.read_csv(csv_file)

# Padding function
def pad_sequence(seq, template_len):
    return seq.ljust(template_len, "-")[:template_len]

df["aligned_imgt_padded"] = df["aligned_imgt"].apply(
    lambda x: pad_sequence(x.strip('"').strip(), length)
)

# ===== AntiBERTy Embeddings =====
try:
    from antiberty import AntiBERTyRunner
except ImportError:
    !pip install antiberty -q
    from antiberty import AntiBERTyRunner

antiberty = AntiBERTyRunner()
emb_size = 512  # AntiBERTy fixed embedding dimension

# Embedding function (corrected)
def embed_sequence(seq):
    mask = torch.tensor([aa != "-" for aa in seq], device=processor)
    seq_no_gaps = "".join([aa for aa in seq if aa != "-"])

    if len(seq_no_gaps) == 0:
        per_residue = torch.zeros((length, emb_size))
        per_sequence = torch.zeros((emb_size,))
        return per_residue, per_sequence

    with torch.no_grad():
        emb_tensor = antiberty.embed([seq_no_gaps])[0]          # (L+2, 512)
        residue_emb = emb_tensor[1:-1, :].to(processor)         # (L, 512)

    per_residue = torch.zeros((length, emb_size), device=processor)
    per_residue[mask] = residue_emb
    per_sequence = per_residue[mask].mean(0)

    del emb_tensor, residue_emb
    gc.collect()
    return per_residue.cpu(), per_sequence.cpu()


# Run embedding
N = len(df)
X_residue = np.zeros((N, length, emb_size), dtype=np.float32)
X_sequence = np.zeros((N, emb_size), dtype=np.float32)
antibody_ids = []

for i, (ab_id, seq) in enumerate(zip(df["antibody_id"], df["aligned_imgt_padded"])):
    print(f"Embedding {i+1}/{N}: {ab_id}")
    per_res, per_seq = embed_sequence(seq)
    X_residue[i] = per_res.numpy()
    X_sequence[i] = per_seq.numpy()
    antibody_ids.append(ab_id)

# Save outputs
os.makedirs(output_dir, exist_ok=True)
np.save(os.path.join(output_dir, f"{prefix}_per_residue.npy"), X_residue)
np.save(os.path.join(output_dir, f"{prefix}_per_sequence.npy"), X_sequence)
np.save(os.path.join(output_dir, f"{prefix}_antibody_ids.npy"), np.array(antibody_ids))
np.save(os.path.join(output_dir, f"{prefix}_imgt_positions.npy"), np.arange(1, length + 1))

print("Embeddings for VL domain saved successfully")

Embedding 1/208: GDPa1-001
Embedding 2/208: GDPa1-002
Embedding 3/208: GDPa1-003
Embedding 4/208: GDPa1-004
Embedding 5/208: GDPa1-005
Embedding 6/208: GDPa1-006
Embedding 7/208: GDPa1-007
Embedding 8/208: GDPa1-008
Embedding 9/208: GDPa1-010
Embedding 10/208: GDPa1-011
Embedding 11/208: GDPa1-012
Embedding 12/208: GDPa1-014
Embedding 13/208: GDPa1-015
Embedding 14/208: GDPa1-016
Embedding 15/208: GDPa1-017
Embedding 16/208: GDPa1-018
Embedding 17/208: GDPa1-019
Embedding 18/208: GDPa1-020
Embedding 19/208: GDPa1-021
Embedding 20/208: GDPa1-022
Embedding 21/208: GDPa1-023
Embedding 22/208: GDPa1-024
Embedding 23/208: GDPa1-027
Embedding 24/208: GDPa1-028
Embedding 25/208: GDPa1-029
Embedding 26/208: GDPa1-030
Embedding 27/208: GDPa1-031
Embedding 28/208: GDPa1-032
Embedding 29/208: GDPa1-033
Embedding 30/208: GDPa1-034
Embedding 31/208: GDPa1-035
Embedding 32/208: GDPa1-036
Embedding 33/208: GDPa1-037
Embedding 34/208: GDPa1-038
Embedding 35/208: GDPa1-040
Embedding 36/208: GDPa1-042
E

In [7]:
# @title Confirm shape of antiBERTy embeddings

# ===== Per-residue =====

# Load embeddings
X_vh = np.load("/content/drive/MyDrive/antiberty_antibody_results/vh_antiberty_per_residue.npy")
X_vl = np.load("/content/drive/MyDrive/antiberty_antibody_results/vl_antiberty_per_residue.npy")

# Display shape of embeddings
print("VH (per-residue) shape:", X_vh.shape)
print("VL (per-residue) shape:", X_vl.shape)

# ===== Per-sequence =====

# Load embeddings
X_vh = np.load("/content/drive/MyDrive/antiberty_antibody_results/vh_antiberty_per_sequence.npy")
X_vl = np.load("/content/drive/MyDrive/antiberty_antibody_results/vl_antiberty_per_sequence.npy")

# Display shape of embeddings
print("VH (per-sequence) shape:", X_vh.shape)
print("VL (per-sequence) shape:", X_vl.shape)

VH (per-residue) shape: (208, 127, 512)
VL (per-residue) shape: (208, 127, 512)
VH (per-sequence) shape: (208, 512)
VL (per-sequence) shape: (208, 512)


In [8]:
# @title Merge antiBERTy embeddings

# ===== Load embeddings =====

# Load embeddings
X_vh = np.load("/content/drive/MyDrive/antiberty_antibody_results/vh_antiberty_per_residue.npy")
Xseq_vh = np.load("/content/drive/MyDrive/antiberty_antibody_results/vh_antiberty_per_sequence.npy")
ids_vh = np.load("/content/drive/MyDrive/antiberty_antibody_results/vh_antiberty_antibody_ids.npy")

X_vl = np.load("/content/drive/MyDrive/antiberty_antibody_results/vl_antiberty_per_residue.npy")
Xseq_vl = np.load("/content/drive/MyDrive/antiberty_antibody_results/vl_antiberty_per_sequence.npy")
ids_vl = np.load("/content/drive/MyDrive/antiberty_antibody_results/vl_antiberty_antibody_ids.npy")

# Build index maps
vh_map = {ab_id: i for i, ab_id in enumerate(ids_vh)}
vl_map = {ab_id: i for i, ab_id in enumerate(ids_vl)}


# ===== Pair and merge embeddings =====

# Pair antibodies
paired_ids = sorted(set(ids_vh).intersection(ids_vl))
print(f"Paired antibodies: {len(paired_ids)}")


# Merge per-residue embeddings
N = len(paired_ids)
emb_size = X_vh.shape[2]

X_paired = np.zeros((N, 254, emb_size), dtype=np.float32)

for i, ab_id in enumerate(paired_ids):
    X_paired[i] = np.concatenate(
        [X_vh[vh_map[ab_id]], X_vl[vl_map[ab_id]]],
        axis=0
    )

# Merge per-sequence embeddings
embed_size = Xseq_vh.shape[1]

Xseq_paired = np.zeros((N, embed_size * 2), dtype=np.float32)

for i, ab_id in enumerate(paired_ids):
    Xseq_paired[i] = np.concatenate(
        [Xseq_vh[vh_map[ab_id]], Xseq_vl[vl_map[ab_id]]],
        axis=0
    )


# ===== Save embeddings =====
output_dir = "/content/drive/MyDrive/antiberty_antibody_results"
np.save(os.path.join(output_dir, "paired_vh_vl_per_residue.npy"), X_paired)
np.save(os.path.join(output_dir, "paired_antibody_ids.npy"), np.array(paired_ids))

# Save per-sequence embeddings
np.save(os.path.join(output_dir, "paired_vh_vl_per_sequence.npy"), Xseq_paired)

# Confirm and display pairing
print("Paired per-residue shape:", X_paired.shape)
print("Paired per-sequence shape:", Xseq_paired.shape)

Paired antibodies: 208
Paired per-residue shape: (208, 254, 512)
Paired per-sequence shape: (208, 1024)
